In [1]:
%pip install openai tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

MODEL="llama-3.1-8b-instant"

In [3]:

def num_tokens_from_messages(messages, model=MODEL):
    """
    Counts the number of tokens used by a structured list of messages.
    """
    try:
        # cl110k_base works good with our model
        encoding = tiktoken.get_encoding("cl100k_base")
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    
    # formatting
    tokens_per_message = 3
    tokens_per_name = 1
    
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
                
    num_tokens += 3  # Array execution wrapper overhead (<|start|>assistant...)
    return num_tokens

def main():
    
    article_headings = [
        "I. Introduction to the 2008 Financial Crisis",
        "II. Historical Background: The Creation of the Housing Bubble",
        "III. Key Players: Financial Institutions and Borrowers",
        "IV. The Domino Effect on the Global Financial System"
    ]
    
    # give context and role to the assistant
    system_prompt = (
        "You are a helpful assistant for a financial news website writing an article series.\n"
        "All of the subheadings you are managing:\n"
    )
    for heading in article_headings:
        system_prompt += f"- {heading}\n"
        
    messages = []
    # developer or system both work the same role
    messages.append({"role": "system", "content": system_prompt})
    
    # set a low maximum token context cap 
    MAX_TOKEN_SIZE = 1200
    all_stored_responses = []
    
    print(f"Initial setup complete. Starting loop with a strict token cap of: {MAX_TOKEN_SIZE}\n")

    # make the multi turn loop
    for heading in article_headings:
        print(f"🚀 Processing: {heading}")
        
        # append new user message to the history
        messages.append({
            "role": "user", 
            "content": f"Write a very large paragraph about {heading}. Make it very long and detailed."
        })
        
    
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        
        # extract the assistant response and store it
        assistant_text = response.choices[0].message.content
        all_stored_responses.append(assistant_text)
        
        # update history
        messages.append({"role": "assistant", "content": assistant_text})
        
        print(f"-> Current message array length: {len(messages)}")
        print(f"-> Calculated size BEFORE trim check: {num_tokens_from_messages(messages)} tokens")
        
        # trim history when it hits max limit
        while num_tokens_from_messages(messages) > MAX_TOKEN_SIZE:
            # find odest non system message to remove
            non_system_msg_index = next(
                (i for i, msg in enumerate(messages) if msg["role"] not in ["system", "developer"]),
                None
            )
            
            if non_system_msg_index is not None:
                # remove oldest turn
                removed = messages.pop(non_system_msg_index)
                print(f"⚠️ [SLIDING WINDOW EFFECT]: Dropped oldest context message ('{removed['role']}') to fit token budget.")
            else:
                # safe break if system message exceed token limit
                break
                
        print(f"-> Final optimized size AFTER validation check: {num_tokens_from_messages(messages)} tokens")
        print(f"-> Groq API verified usage metrics -> Input tokens: {response.usage.prompt_tokens} | Output tokens: {response.usage.completion_tokens}\n")

    print("--- Session Finished Successfully ---")
    print(f"Total compiled sections generated: {len(all_stored_responses)}")

if __name__ == "__main__":
    main()

Initial setup complete. Starting loop with a strict token cap of: 1200

🚀 Processing: I. Introduction to the 2008 Financial Crisis
-> Current message array length: 3
-> Calculated size BEFORE trim check: 1014 tokens
-> Final optimized size AFTER validation check: 1014 tokens
-> Groq API verified usage metrics -> Input tokens: 132 | Output tokens: 902

🚀 Processing: II. Historical Background: The Creation of the Housing Bubble
-> Current message array length: 5
-> Calculated size BEFORE trim check: 2227 tokens
⚠️ [SLIDING WINDOW EFFECT]: Dropped oldest context message ('user') to fit token budget.
⚠️ [SLIDING WINDOW EFFECT]: Dropped oldest context message ('assistant') to fit token budget.
⚠️ [SLIDING WINDOW EFFECT]: Dropped oldest context message ('user') to fit token budget.
⚠️ [SLIDING WINDOW EFFECT]: Dropped oldest context message ('assistant') to fit token budget.
-> Final optimized size AFTER validation check: 81 tokens
-> Groq API verified usage metrics -> Input tokens: 1068 | Ou